<a target="_blank" href="https://colab.research.google.com/github/cohere-ai/cohere-developer-experience/blob/main/notebooks/guides/Analyzing_Hacker_News_with_Six_Language_Understanding_Methods.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Analyzing Hacker News with Six Language Understanding Methods
Large language models give machines a vastly improved representation and understanding of language. These abilities give developers more options for content recommendation, analysis, and filtering.

In this notebook we take thousands of the most popular posts from Hacker News and demonstrate some of these functionalities:

1. Given an existing post title, retrieve the most similar posts (nearest neighbor search using embeddings)
1. Given a query that we write, retrieve the most similar posts
1. Plot the archive of articles by similarity (where similar posts are close together and different ones are far)
1. Cluster the posts to identify the major common themes
1. Extract major keywords from each cluster so we can identify what the clsuter is about
1. (Experimental) Name clusters with a generative language model

## Setup
Let's start by installing the tools we'll need and then importing them.

In [1]:
!pip install cohere umap-learn altair annoy bertopic

In [2]:
import cohere
import numpy as np
import pandas as pd
import umap
import altair as alt
from annoy import AnnoyIndex
import warnings
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)

Fill in your Cohere API key in the next cell. To do this, begin by [signing up to Cohere](https://os.cohere.ai/) (for free!) if you haven't yet. Then get your API key [here](https://dashboard.cohere.com/api-keys).

In [3]:
# Paste your API key here. Remember to not share publicly
co = cohere.Client("COHERE_API_KEY") # Insert your Cohere API key

## Dataset: Top 3,000 Ask HN posts
We will use the top 3,000 posts from the Ask HN section of Hacker News. We provide a CSV containing the posts.

In [7]:
# Load the dataframe containing the text and metadata of the posts
df = pd.read_csv('https://storage.googleapis.com/cohere-assets/blog/text-clustering/data/askhn3k_df.csv', index_col=0)

print(f'Loaded a DataFrame with {len(df)} rows')

Loaded a DataFrame with 3000 rows


In [8]:
# Let's glance at the contents of the dataframe with the text and metadata
df.head()

,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,"I'm a software engineer going blind, how should I prepare?",NaN,"I&#x27;m a 24 y&#x2F;o full stack engineer (I know some of you are rolling your eyes right now, just highlighting that I have experience on frontend apps as well as backend architecture). I&#x27;ve been working professionally for ~7 years building mostly javascript projects but also some PHP. Two years ago I was diagnosed with a condition called &quot;Usher&#x27;s Syndrome&quot; - characterized by hearing loss, balance issues, and progressive vision loss.<p>I know there are blind software engineers out there. My main questions are:<p>- Are there blind frontend engineers?<p>- What kinds of software engineering lend themselves to someone with limited vision? Backend only?<p>- Besides a screen reader, what are some of the best tools for building software with limited vision?<p>- Does your company employ blind engineers? How well does it work? What kind of engineer are they?<p>I&#x27;m really trying to get ahead of this thing and prepare myself as my vision is degrading rather quickly. I&#x27;m not sure what I can do if I can&#x27;t do SE as I don&#x27;t have any formal education in anything. I&#x27;ve worked really hard to get to where I am and don&#x27;t want it to go to waste.<p>Thank you for any input, and stay safe out there!<p>Edit:<p>Thank you all for your links, suggestions, and moral support, I really appreciate it. Since my diagnosis I&#x27;ve slowly developed a crippling anxiety centered around a feeling that I need to figure out the rest of my life before it&#x27;s too late. I know I shouldn&#x27;t think this way but it is hard not to. I&#x27;m very independent and I feel a pressure to &quot;show up.&quot; I will look into these opportunities mentioned and try to get in touch with some more members of the blind engineering community.",NaN,zachrip,3270,1587332026,2020-04-19 21:33:46+00:00,story,22918980,NaN,473.0,NaN,NaN
1,Am I the longest-serving programmer – 57 years and counting?,NaN,"In May of 1963, I started my first full-time job as a computer programmer for Mitchell Engineering Company, a supplier of steel buildings. At Mitchell, I developed programs in Fortran II on an IBM 1620 mostly to improve the efficiency of order processing and fulfillment. Since then, all my jobs for the past 57 years have involved computer programming. I am now a data scientist developing cloud-based big data fraud detection algorithms using machine learning and other advanced analytical technologies. Along the way, I earned a Master’s in Operations Research and a Master’s in Management Science, studied artificial intelligence for 3 years in a Ph.D. program for engineering, and just two years ago I received Graduate Certificates in Big Data Analytics from the schools of business and computer science at a local university (FAU). In addition, I currently hold the designation of Certified Analytics Professional (CAP). At 74, I still have no plans to retire or to stop programming.",NaN,genedangelo,2634,1590890024,2020-05-31 01:53:44+00:00,story,23366546,NaN,531.0,NaN,NaN
2,Is S3 down?,NaN,I&#x27;m getting<p>{\n &quot;errorCode&quot; : &quot;InternalError&quot;\n}<p>When I attempt to use the AWS Console to view s3,NaN,iamdeedubs,2589,1488303958,2017-02-28 17:45:58+00:00,story,13755673,NaN,1055.0,NaN,NaN
3,What tech job would let me get away with the least real work possible?,NaN,"Hey HN,<p>I&#x27;ll probably get a lot of flak for this. Sorry.<p>I&#x27;m an average developer looking for ways to work as little as humanely possible.<p>The pandemic made me realize that I do not care about working anymore. The software I build is useless. Time flies real fast and I have to focus on my passions (which are not monetizable).<p>Unfortunately, I require shelter, calories and hobby materials. Thus the need for some kind of job.<p>Which leads me to ask my fellow tech workers, what kind of job (if any)

We calculate the embeddings using Cohere's `embed-v4.0` model. The resulting embeddings matrix has 3,000 rows (one for each post) and 1024 columns (meaning each post title is represented with a 1024-dimensional embedding).

In [10]:
batch_size = 90

embeds_list = []
for i in range(0, len(df), batch_size):
    batch = df[i : min(i + batch_size, len(df))]
    texts = list(batch["title"])
    embs_batch = co.embed(
        texts=texts, model="embed-v4.0", input_type="search_document"
    ).embeddings
    embeds_list.extend(embs_batch)

embeds = np.array(embeds_list)
embeds.shape

(3000, 1024)

## Building a semantic search index
For nearest-neighbor search, we can use the open-source Annoy library. Let's create a semantic search index and feed it all the embeddings.

In [11]:
# Create the search index, pass the size of embedding
search_index = AnnoyIndex(embeds.shape[1], 'angular')
# Add all the vectors to the search index
for i in range(len(embeds)):
    search_index.add_item(i, embeds[i])

search_index.build(10) # 10 trees
search_index.save('askhn.ann')

True

## 1- Given an existing post title, retrieve the most similar posts (nearest neighbor search using embeddings)
We can query neighbors of a specific post using `get_nns_by_item`.

In [13]:
# Choose an example (we'll retrieve others similar to it)
example_id = 50

# Retrieve nearest neighbors
similar_item_ids = search_index.get_nns_by_item(example_id,
                                                10, # Number of results to retrieve
                                                include_distances=True)
# Format and print the text and distances
results = pd.DataFrame(data={'post titles': df.iloc[similar_item_ids[0]]['title'],
                             'distance': similar_item_ids[1]}).drop(example_id)

print(f"Query post:'{df.iloc[example_id]['title']}'\nNearest neighbors:")
results

Query post:'Pick startups for YC to fund'
Nearest neighbors:


,post titles,distance
2991,Best Bank for Startups?,0.883494
2910,Who's looking for a cofounder?,0.885087
31,What startup/technology is on your 'to watch' list?,0.887212
685,What startup/technology is on your 'to watch' list?,0.887212
2123,Who is seeking a cofounder?,0.889451
727,Agriculture startups doing interesting work?,0.899192
2972,How should I evaluate a startup as I job hunt?,0.901621
2589,What methods do you use to gain early customers for your startup?,0.903065
2708,Is there VC appetite for defense related startups?,0.904016


## 2- Given a query that we write, retrieve the most similar posts
We're not limited to searching using existing items. If we get a query, we can embed it and find its nearest neighbors from the dataset.

In [14]:
query = "How can I improve my knowledge of calculus?"

# Get the query's embedding
# We'll need to embed the query using the same model that we used to embed the post titles
# so the query and titles are using the same embedding space.
query_embed = co.embed(texts=[query],
                       model="embed-v4.0",
                       truncate="RIGHT",
                       input_type="search_query").embeddings

# Retrieve the nearest neighbors
similar_item_ids = search_index.get_nns_by_vector(query_embed[0], 10, include_distances=True)

# Format & print the results
results = pd.DataFrame(data={'texts': df.iloc[similar_item_ids[0]]['title'],
                             'distance': similar_item_ids[1]})
print(f"Query:'{query}'\nNearest neighbors:")
results

Query:'How can I improve my knowledge of calculus?'
Nearest neighbors:


,texts,distance
2457,How do I improve my command of mathematical language?,0.931286
1235,How to learn new things better?,1.024635
145,How to self-learn math?,1.044135
1317,How can I learn to read mathematical notation?,1.050976
910,How Do You Learn?,1.061253
2432,How did you learn math notation?,1.070800
1994,How do I become smarter?,1.083434
1529,How do you personally learn?,1.086088
796,How do you keep improving?,1.087251
1286,How do I learn drawing?,1.088468


## 3- Plot the archive of articles by similarity
What if we want to browse the archive instead of only searching it? Let's plot all the questions in a 2D chart so you're able to visualize the posts in the archive and their similarities.

In [15]:
reducer = umap.UMAP(n_neighbors=100)
umap_embeds = reducer.fit_transform(embeds)

In [16]:
df['x'] = umap_embeds[:,0]
df['y'] = umap_embeds[:,1]

# Plot
chart = alt.Chart(df).mark_circle(size=60).encode(
    x=#'x',
    alt.X('x',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(labels=False, ticks=False, domain=False)
    ),
    y=
    alt.Y('y',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(labels=False, ticks=False, domain=False)
    ),
    tooltip=['title']
    ).configure(background="#FDF7F0"
    ).properties(
        width=700,
        height=400,
        title='Ask HN: top 3,000 posts'
        )

chart.interactive()

alt.Chart(...)

## 4- Cluster the posts to identify the major common themes
Let's proceed to cluster the embeddings using KMeans from scikit-learn.

In [17]:
# Pick the number of clusters
n_clusters = 8

# Cluster the embeddings
kmeans_model = KMeans(n_clusters=n_clusters, random_state=0)
classes = kmeans_model.fit_predict(embeds)

## 5- Extract major keywords from each cluster so we can identify what the cluster is about

In [18]:
# Extract the keywords for each cluster
documents =  df['title']
documents = pd.DataFrame({"Document": documents,
                          "ID": range(len(documents)),
                          "Topic": None})
documents['Topic'] = classes
documents_per_topic = documents.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
count_vectorizer = CountVectorizer(stop_words="english").fit(documents_per_topic.Document)
count = count_vectorizer.transform(documents_per_topic.Document)
words = count_vectorizer.get_feature_names_out()

In [19]:
ctfidf = ClassTfidfTransformer().fit_transform(count).toarray()
words_per_class = {label: [words[index] for index in ctfidf[label].argsort()[-10:]] for label in documents_per_topic.Topic}
df['cluster'] = classes
df['keywords'] = df['cluster'].map(lambda topic_num: ", ".join(np.array(words_per_class[topic_num])[:]))

## Plot with clusters and keywords information
We can now plot the documents with their clusters and keywords

In [20]:
selection = alt.selection_multi(fields=['keywords'], bind='legend')

chart = alt.Chart(df).transform_calculate(
    url='https://news.ycombinator.com/item?id=' + alt.datum.id
).mark_circle(size=60, stroke='#666', strokeWidth=1, opacity=0.3).encode(
    x=#'x',
    alt.X('x',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(labels=False, ticks=False, domain=False)
    ),
    y=
    alt.Y('y',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(labels=False, ticks=False, domain=False)
    ),
    href='url:N',
    color=alt.Color('keywords:N',
                    legend=alt.Legend(columns=1, symbolLimit=0, labelFontSize=14)
                   ),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=['title', 'keywords', 'cluster', 'score', 'descendants']
).properties(
    width=800,
    height=500
).add_selection(
    selection
).configure_legend(labelLimit= 0).configure_view(
    strokeWidth=0
).configure(background="#FDF7F0").properties(
    title='Ask HN: Top 3,000 Posts'
)
chart.interactive()

alt.Chart(...)

## 6- (Experimental) Naming clusters with a generative language model
While the extracted keywords do add a lot of information to help us identify the clusters at a glance, we should be able to have a generative model look at these keywords and suggest a name. So far I have reasonable results from a prompt that looks like this:

```
The common theme of the following words: books, book, read, the, you, are, what, best, in, your
is that they all relate to favorite books to read.
---
The common theme of the following words: startup, company, yc, failed
is that they all relate to startup companies and their failures.
---
The common theme of the following words: freelancer, wants, hired, be, who, seeking, to, 2014, 2020, april
is that they all relate to hiring for a freelancer to join the team of a startup.
---
The common theme of the following words: <insert keywords here>
is that they all relate to
```

There's a lot of room for improvement though. I'm really excited by this use case because it adds so much information. Imagine if the in the following tree of topics, you assigned each cluster an intelligible name. Then imagine if you assigned each *branching* a name as well

![](https://raw.githubusercontent.com/cohere-ai/cohere-developer-experience/main/notebooks/images/kmeans-centroid-dendrogram.png)

We can’t wait to see what you start building! Share your projects or find support on our [Discord server](https://discord.com/invite/co-mmunity).
